# ms-mint Demo

This notebook demonstrates ms-mint for targeted metabolomics data processing.

**Features:**
- Load MS files (mzML, mzXML)
- Load target lists and metadata
- Run peak integration
- Visualizations (heatmaps, PCA, peak shapes)
- Export results

For more information, see the [ms-mint documentation](https://lewisresearchgroup.github.io/ms-mint).

## 1. Installation

In [ ]:
!pip install -q "ms-mint @ git+https://github.com/LewisResearchGroup/ms-mint.git@feature/jupyter-gui-improvements"

## 2. Download Demo Data

The demo dataset contains 12 LC-MS files from three bacterial species:
- **EC**: *Escherichia coli* (4 replicates)
- **SA**: *Staphylococcus aureus* (4 replicates)
- **CA**: *Candida albicans* (4 replicates)

Also includes a target list with 7 metabolites and sample metadata.

In [ ]:
from pathlib import Path
import zipfile

# Download demo data from Zenodo
print("Downloading demo data from Zenodo...")
!wget -q --show-progress -O demo_data.zip "https://zenodo.org/records/17727891/files/230508__ms-mint-demofiles.zip?download=1"

# Extract
with zipfile.ZipFile("demo_data.zip", "r") as z:
    z.extractall(".")

# List downloaded files
ms_files = sorted(Path("ms-files").glob("*"))
print(f"\nExtracted {len(ms_files)} MS files:")
for f in ms_files:
    print(f"  {f}")

## 3. Load Data

Load MS files, target list, and metadata into a Mint instance. The target list defines which metabolites to extract (m/z and retention time windows).

In [ ]:
from ms_mint.notebook import Mint

# Create a Mint instance
mint = Mint(verbose=True)

# Load demo data
mint.load_files('ms-files/*')
mint.load_targets('targets.csv')
mint.load_metadata('metadata.csv')

# Show metadata
mint.meta

## 4. Run Processing

Extract peak intensities for all targets across all MS files.

In [ ]:
# Run processing
mint.run()

### Peak Optimization
Optimize retention time windows for better peak detection. This refines `rt_min` and `rt_max` values.

In [ ]:
# Optimize RT window for Succinate
mint.opt.rt_min_max(peak_labels=['Succinate'], plot=True)

## 5. View Results

Results table contains peak areas, heights, retention times, and quality metrics for each target in each file.

In [ ]:
mint.results

## 6. Visualizations

### Peak Shapes
Overlay chromatographic peaks for each target across all samples to assess peak quality and retention time consistency.

In [ ]:
# Peak shapes
mint.plot.peak_shapes(col_wrap=3)

### Hierarchical Clustering
Cluster samples by their metabolite profiles using agglomerative hierarchical clustering with cosine distance.

In [ ]:
# Hierarchical clustering
mint.plot.hierarchical_clustering(figsize=(10, 8));

### Chromatograms
Extract and plot ion chromatograms for specific m/z values.

In [ ]:
# Chromatogram for Succinate (m/z 117.019)
mint.plot.chromatogram(mz=117.019, mz_width=10)

### Principal Component Analysis (PCA)
Reduce dimensionality and visualize sample groupings. Samples should cluster by organism (EC, SA, CA).

In [ ]:
# PCA
mint.pca.run(n_components=3)
mint.pca.plot.pairplot(n_components=3, hue='label')

## 7. Resources

- [Documentation](https://lewisresearchgroup.github.io/ms-mint)
- [GitHub Repository](https://github.com/LewisResearchGroup/ms-mint)
- [Demo Data](https://zenodo.org/records/17727891)
- [Report Issues](https://github.com/LewisResearchGroup/ms-mint/issues)